# 03 - Sensor de Amonia
> Seletividade quimica, firmware de leitura e calibracao

**Modulo:** `13_enfitec_projetos` | **Feito com:** [Claude](https://claude.ai) (Anthropic)

---


In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
HAIKU  = 'claude-haiku-4-5-20251001'
SONNET = 'claude-sonnet-4-5'

def ask(prompt, system=None, model=HAIKU, max_tokens=1024):
    kw = dict(model=model, max_tokens=max_tokens,
              messages=[{'role':'user','content':prompt}])
    if system: kw['system'] = system
    return client.messages.create(**kw).content[0].text

print('Pronto!')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
print('numpy e matplotlib importados!')

## Principios de Deteccao de Amonia

Vamos pedir ao Claude para explicar os mecanismos de deteccao
e nos ajudar a escolher a melhor tecnologia para nossa aplicacao.

In [ ]:
prompt_principios = """Explique os tres principais mecanismos de deteccao de
amonia (NH3) para sensores industriais:

1. SENSOR MOS (Metal Oxide Semiconductor):
   - Principio de funcionamento (variacao de resistencia)
   - Materiais tipicos (SnO2, WO3, ZnO)
   - Faixa de deteccao, tempo de resposta, vida util
   - Vantagens e desvantagens

2. SENSOR ELETROQUIMICO:
   - Principio de funcionamento (corrente proporcional)
   - Eletrodos e eletrolito
   - Faixa de deteccao, seletividade
   - Vantagens e desvantagens

3. SENSOR OPTICO (NDIR / Absorpcao UV):
   - Principio de funcionamento (absorcao de luz)
   - Comprimentos de onda para NH3
   - Precisao e faixa de deteccao
   - Vantagens e desvantagens

Para uma aplicacao em AVIARIO (deteccao de amonia 0-100 ppm,
ambiente umido, temperatura 15-35C, custo baixo), qual tecnologia
voce recomenda e por que?"""

resp_principios = ask(prompt_principios, model=SONNET, max_tokens=2048)
print(resp_principios)

In [ ]:
# Comparativo das tecnologias
tecnologias = {
    'MOS (SnO2)':       {'faixa': '1-1000 ppm', 'precisao': '+-10%',
                         'tempo_resp': '10-30s', 'vida_util': '2-5 anos',
                         'custo': 'Baixo ($2-10)', 'consumo': '150-800 mW'},
    'Eletroquimico':    {'faixa': '0-500 ppm', 'precisao': '+-2%',
                         'tempo_resp': '30-60s', 'vida_util': '1-3 anos',
                         'custo': 'Medio ($15-50)', 'consumo': '<1 mW'},
    'Optico (NDIR)':    {'faixa': '0-10000 ppm', 'precisao': '+-1%',
                         'tempo_resp': '1-5s', 'vida_util': '5-10 anos',
                         'custo': 'Alto ($100-500)', 'consumo': '50-200 mW'},
}

print('Comparativo de Tecnologias de Deteccao de NH3')
print('=' * 70)
for tech, specs in tecnologias.items():
    print(f'\n--- {tech} ---')
    for param, valor in specs.items():
        print(f'  {param:15s}: {valor}')

print('\n' + '=' * 70)
print('Recomendacao para aviario: Sensor MOS (SnO2/WO3)')
print('Motivo: custo baixo, faixa adequada, boa disponibilidade')

## Seletividade e Interferentes

Vamos pedir ao Claude para analisar a sensibilidade cruzada
com outros gases e sugerir estrategias de mitigacao.

In [ ]:
prompt_seletividade = """Para um sensor MOS de amonia (SnO2 dopado com Pt),
analise a sensibilidade cruzada com os seguintes gases interferentes
comuns em ambientes de aviarios:

1. CO (monoxido de carbono) - de aquecedores
2. CO2 (dioxido de carbono) - respiracao das aves
3. H2S (sulfeto de hidrogenio) - decomposicao de materia organica
4. CH4 (metano) - fermentacao de cama de frango
5. Vapor de agua (umidade relativa 40-90%)

Para cada interferente, informe:
- Nivel tipico de sensibilidade cruzada (% do sinal de NH3)
- Concentracao tipica no ambiente
- Impacto na leitura de NH3

Sugira estrategias de mitigacao:
- Filtros quimicos (carvao ativado, etc.)
- Compensacao por software (temperatura, umidade)
- Array de sensores (nariz eletronico)
- Algoritmos de separacao de sinais"""

resp_seletividade = ask(prompt_seletividade, model=SONNET, max_tokens=2048)
print(resp_seletividade)

In [ ]:
# Visualizacao da sensibilidade cruzada
gases = ['NH3\n(alvo)', 'CO', 'CO2', 'H2S', 'CH4', 'H2O']
sensibilidade_relativa = [1.0, 0.15, 0.02, 0.35, 0.08, 0.25]
cores = ['#2196F3', '#FF9800', '#4CAF50', '#F44336', '#9C27B0', '#00BCD4']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Grafico de barras - sensibilidade relativa
bars = ax1.bar(gases, sensibilidade_relativa, color=cores, edgecolor='black')
ax1.set_ylabel('Sensibilidade Relativa')
ax1.set_title('Sensibilidade Cruzada do Sensor MOS (SnO2)')
ax1.axhline(y=0.1, color='r', linestyle='--', alpha=0.5, label='Limiar aceitavel (10%)')
ax1.legend()
for bar, val in zip(bars, sensibilidade_relativa):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
             f'{val*100:.0f}%', ha='center', fontsize=9)

# Grafico radar - impacto no aviario
categorias = ['Seletividade', 'Estabilidade', 'Tempo Resp.',
              'Durabilidade', 'Custo', 'Precisao']
valores_mos = [0.6, 0.5, 0.7, 0.7, 0.9, 0.6]
valores_eletro = [0.9, 0.6, 0.5, 0.4, 0.6, 0.85]

angulos = np.linspace(0, 2 * np.pi, len(categorias), endpoint=False).tolist()
valores_mos += valores_mos[:1]
valores_eletro += valores_eletro[:1]
angulos += angulos[:1]

ax2 = fig.add_subplot(122, polar=True)
ax2.plot(angulos, valores_mos, 'o-', linewidth=2, label='MOS', color='#2196F3')
ax2.fill(angulos, valores_mos, alpha=0.15, color='#2196F3')
ax2.plot(angulos, valores_eletro, 's-', linewidth=2, label='Eletroquimico', color='#F44336')
ax2.fill(angulos, valores_eletro, alpha=0.15, color='#F44336')
ax2.set_xticks(angulos[:-1])
ax2.set_xticklabels(categorias, fontsize=8)
ax2.set_title('Comparativo MOS vs Eletroquimico', pad=20)
ax2.legend(loc='lower right', fontsize=8)

plt.tight_layout()
plt.savefig('seletividade_sensor.png', dpi=100, bbox_inches='tight')
plt.show()
print('Analise de seletividade concluida!')

## Circuito de Leitura para Sensor MOS

Vamos pedir ao Claude para projetar o circuito de interface
entre o sensor MOS e o microcontrolador.

In [ ]:
prompt_circuito = """Projete o circuito completo de leitura para um sensor MOS
de amonia (tipo MQ-137 ou equivalente) conectado a um ESP32. Inclua:

1. DIVISOR DE TENSAO:
   - Resistor de carga (RL) dimensionado para faixa 10-200 ppm
   - Calculo de RL otimo para maxima sensibilidade
   - Tensao de saida vs concentracao

2. CONTROLE DO AQUECEDOR:
   - Circuito com MOSFET para controlar o heater (5V, ~150mA)
   - PWM para controle de temperatura do heater
   - Ciclo de aquecimento (pre-heat de 24h, operacao normal)

3. CONDICIONAMENTO DE SINAL:
   - Filtro RC anti-aliasing antes do ADC
   - Amplificador operacional (se necessario)
   - Protecao contra sobretensao no ADC (3.3V max)

4. INTERFACE COM ADC do ESP32:
   - Pino ADC recomendado (ADC1 para evitar conflito com WiFi)
   - Resolucao e faixa de leitura
   - Oversampling para aumentar resolucao efetiva

Desenhe o esquematico em formato texto/ASCII e forneca a lista de
componentes (BOM) com valores e especificacoes."""

resp_circuito = ask(prompt_circuito, model=SONNET, max_tokens=2048)
print(resp_circuito)

In [ ]:
# Esquematico em texto e analise do divisor de tensao
esquematico = """
=======================================================================
     CIRCUITO DE LEITURA - SENSOR MOS NH3 (MQ-137)
=======================================================================

    +5V (VCC)                        +3.3V (ESP32)
     |                                |
     |                                |
    [HEATER]                         [RL = 10k]
     |  (Rh ~33 ohm)                  |
     |                                +----> Vo (para ADC)
     |                                |        |
    [Q1 MOSFET]                      [Rs]     [R = 10k]
     |  IRLZ44N                       |        |
     |                                |       [C = 100nF]
     |                                |        |
    GND                              GND      GND
     |                                   (filtro RC)
    [ESP32 GPIO 25]
    (PWM heater)

    ESP32 ADC1_CH6 (GPIO 34) <--- Vo (0 - 3.3V)

    Vo = 3.3V * RL / (RL + Rs)
    Rs diminui com aumento de NH3
    Vo aumenta com aumento de NH3

=======================================================================

LISTA DE COMPONENTES (BOM):
  - Sensor MQ-137 (NH3, 5-500 ppm)
  - RL: Resistor 10k ohm 1/4W (divisor de tensao)
  - Q1: IRLZ44N (MOSFET N-channel, logic-level)
  - R_gate: 10k ohm (pull-down gate MOSFET)
  - R_filter: 10k ohm (filtro anti-aliasing)
  - C_filter: 100nF ceramico (filtro anti-aliasing)
  - Frequencia de corte: fc = 1/(2*pi*10k*100n) = 159 Hz
"""
print(esquematico)

# Curva Rs vs concentracao (tipica sensor MOS)
ppm = np.logspace(0, 3, 100)  # 1 a 1000 ppm
# Rs segue lei de potencia: Rs = a * ppm^(-b)
Rs_ar_limpo = 50000  # 50k ohm em ar limpo
a, b = Rs_ar_limpo, 0.42
Rs = a * ppm ** (-b)

RL = 10000  # 10k ohm
Vcc = 3.3
Vo = Vcc * RL / (RL + Rs)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.loglog(ppm, Rs / 1000, 'b-', linewidth=2)
ax1.set_xlabel('Concentracao NH3 (ppm)')
ax1.set_ylabel('Resistencia do Sensor Rs (kohm)')
ax1.set_title('Rs vs Concentracao de NH3')
ax1.grid(True, which='both', alpha=0.3)
ax1.axhline(y=RL / 1000, color='r', linestyle='--', label=f'RL = {RL/1000:.0f}k')
ax1.legend()

ax2.semilogx(ppm, Vo, 'r-', linewidth=2)
ax2.set_xlabel('Concentracao NH3 (ppm)')
ax2.set_ylabel('Tensao de Saida Vo (V)')
ax2.set_title('Tensao de Saida vs Concentracao')
ax2.grid(True, which='both', alpha=0.3)
ax2.axhline(y=3.3, color='gray', linestyle='--', alpha=0.5, label='Vmax ADC (3.3V)')
ax2.axvspan(10, 100, alpha=0.1, color='green', label='Faixa alvo (10-100 ppm)')
ax2.legend()

plt.tight_layout()
plt.savefig('circuito_sensor.png', dpi=100, bbox_inches='tight')
plt.show()
print('Analise do circuito concluida!')

## Firmware de Aquisicao e Calibracao

Vamos pedir ao Claude para gerar firmware completo com leitura
do sensor, compensacao de temperatura e calibracao multi-ponto.

In [ ]:
prompt_firmware = """Gere firmware completo em C/C++ (Arduino framework) para ESP32
que faz aquisicao de dados de um sensor MOS de amonia (MQ-137) com:

1. LEITURA DO ADC:
   - Pino ADC1_CH6 (GPIO 34)
   - Oversampling 16x para resolucao efetiva de 14 bits
   - Media movel de 10 leituras para filtrar ruido
   - Conversao de ADC para tensao e depois para Rs

2. COMPENSACAO DE TEMPERATURA E UMIDADE:
   - Ler sensor DHT22 (temperatura + umidade)
   - Aplicar fator de correcao: Rs_corr = Rs / fator(T, RH)
   - Tabela de correcao do datasheet

3. CALIBRACAO MULTI-PONTO:
   - Armazenar pontos de calibracao na EEPROM/NVS
   - Minimo 3 pontos (0, 25, 50, 100 ppm)
   - Interpolacao por lei de potencia: ppm = a * (Rs/Ro)^b
   - Funcao de calibracao automatica (modo calibracao)

4. CONTROLE DO HEATER:
   - PWM a 1kHz no GPIO 25
   - Ciclo de pre-aquecimento (ramp-up de 48h na primeira vez)
   - Duty cycle ajustavel para controle de temperatura

5. SAIDA DE DADOS:
   - Serial (115200 baud): ppm, Rs, temperatura, umidade, status
   - JSON format para integracao com backend

Use: Arduino framework, DHT.h, Preferences.h (NVS).
Codigo completo com comentarios em portugues."""

resp_firmware = ask(prompt_firmware, model=SONNET, max_tokens=4096)
print(resp_firmware)

In [ ]:
# Codigo de referencia do firmware
firmware_nh3 = '''// =============================================
// Firmware ESP32 - Sensor de Amonia MQ-137
// Com calibracao multi-ponto e compensacao T/RH
// =============================================

#include <Arduino.h>
#include <DHT.h>
#include <Preferences.h>
#include <ArduinoJson.h>

// --- Pinos ---
#define SENSOR_PIN    34   // ADC1_CH6
#define HEATER_PIN    25   // PWM heater
#define DHT_PIN       26   // DHT22
#define LED_PIN       2    // LED status

// --- Constantes ---
#define ADC_RESOLUTION   12
#define OVERSAMPLING     16
#define MEDIA_MOVEL_N    10
#define RL_OHM           10000.0
#define VCC              3.3
#define HEATER_PWM_FREQ  1000
#define HEATER_PWM_CH    0

DHT dht(DHT_PIN, DHT22);
Preferences prefs;

// --- Variaveis de calibracao ---
float Ro = 10000.0;     // Rs em ar limpo (calibrado)
float cal_a = 102.2;    // Coeficiente lei de potencia
float cal_b = -2.473;   // Expoente lei de potencia

// --- Buffer media movel ---
float buffer_rs[10];
int buf_idx = 0;
bool buf_cheio = false;

void setup() {
    Serial.begin(115200);
    analogReadResolution(ADC_RESOLUTION);
    pinMode(LED_PIN, OUTPUT);

    // Configurar PWM do heater
    ledcSetup(HEATER_PWM_CH, HEATER_PWM_FREQ, 8);
    ledcAttachPin(HEATER_PIN, HEATER_PWM_CH);
    ledcWrite(HEATER_PWM_CH, 200);  // ~78% duty

    dht.begin();

    // Carregar calibracao da NVS
    prefs.begin("nh3cal", true);
    Ro = prefs.getFloat("Ro", 10000.0);
    cal_a = prefs.getFloat("cal_a", 102.2);
    cal_b = prefs.getFloat("cal_b", -2.473);
    prefs.end();

    Serial.println("Sensor NH3 inicializado");
    Serial.printf("Ro=%.0f, a=%.2f, b=%.3f\\n", Ro, cal_a, cal_b);
}

float ler_adc_oversampled() {
    uint32_t soma = 0;
    for (int i = 0; i < OVERSAMPLING; i++) {
        soma += analogRead(SENSOR_PIN);
        delayMicroseconds(100);
    }
    return (float)soma / OVERSAMPLING;
}

float adc_para_rs(float adc_val) {
    float voltagem = (adc_val / 4095.0) * VCC;
    if (voltagem < 0.01) voltagem = 0.01;
    float Rs = RL_OHM * (VCC - voltagem) / voltagem;
    return Rs;
}

float compensar_temp_umid(float Rs, float temp, float umid) {
    // Fator de correcao baseado no datasheet
    float fator_t = 1.0 + 0.015 * (temp - 20.0);
    float fator_h = 1.0 + 0.005 * (umid - 65.0);
    return Rs / (fator_t * fator_h);
}

float rs_para_ppm(float Rs) {
    float ratio = Rs / Ro;
    if (ratio < 0.01) ratio = 0.01;
    return cal_a * pow(ratio, cal_b);
}

void loop() {
    float adc = ler_adc_oversampled();
    float Rs_raw = adc_para_rs(adc);

    float temp = dht.readTemperature();
    float umid = dht.readHumidity();
    if (isnan(temp)) temp = 25.0;
    if (isnan(umid)) umid = 50.0;

    float Rs = compensar_temp_umid(Rs_raw, temp, umid);

    // Media movel
    buffer_rs[buf_idx] = Rs;
    buf_idx = (buf_idx + 1) % MEDIA_MOVEL_N;
    if (buf_idx == 0) buf_cheio = true;

    int n = buf_cheio ? MEDIA_MOVEL_N : buf_idx;
    float Rs_media = 0;
    for (int i = 0; i < n; i++) Rs_media += buffer_rs[i];
    Rs_media /= n;

    float ppm = rs_para_ppm(Rs_media);

    // Saida JSON
    StaticJsonDocument<256> doc;
    doc["ppm"] = round(ppm * 10) / 10.0;
    doc["Rs"] = round(Rs_media);
    doc["temp"] = round(temp * 10) / 10.0;
    doc["umid"] = round(umid * 10) / 10.0;
    doc["status"] = (ppm < 25) ? "OK" : (ppm < 50) ? "ATENCAO" : "CRITICO";

    char buf[256];
    serializeJson(doc, buf);
    Serial.println(buf);

    digitalWrite(LED_PIN, !digitalRead(LED_PIN));
    delay(1000);
}
'''

print('Firmware NH3 de referencia carregado.')
print(f'Tamanho: {len(firmware_nh3)} caracteres')
print('\nFuncionalidades:')
print('  - Oversampling 16x (resolucao efetiva ~14 bits)')
print('  - Media movel de 10 leituras')
print('  - Compensacao de temperatura e umidade')
print('  - Calibracao armazenada em NVS (flash)')
print('  - Saida JSON via Serial')

## Curva de Calibracao

Vamos gerar dados sinteticos de resposta do sensor e implementar
ajuste de curva (lei de potencia e logaritmico) com o Claude.

In [ ]:
prompt_calibracao = """Implemente em Python a calibracao de um sensor MOS de amonia.

Dados de calibracao (concentracao conhecida vs Rs/Ro):
  ppm = [5, 10, 20, 50, 100, 200, 500]
  Rs_Ro = [3.5, 2.5, 1.8, 1.1, 0.72, 0.45, 0.22]

Implemente dois modelos de ajuste:
1. Lei de potencia: ppm = a * (Rs/Ro)^b  (linearizar com log-log)
2. Logaritmico: ppm = a * exp(b * Rs/Ro)

Para cada modelo:
- Ajustar os parametros (a, b) usando scipy.optimize.curve_fit
- Calcular R-quadrado
- Calcular erro maximo e erro medio em ppm
- Plotar curva ajustada vs dados de calibracao

Recomendar qual modelo usar e implementar funcao de conversao final.
Use numpy, scipy, matplotlib."""

resp_calibracao = ask(prompt_calibracao, model=SONNET, max_tokens=2048)
print(resp_calibracao)

In [ ]:
from scipy.optimize import curve_fit

# Dados de calibracao (simulados mas realistas)
ppm_cal = np.array([5, 10, 20, 50, 100, 200, 500])
rs_ro_cal = np.array([3.5, 2.5, 1.8, 1.1, 0.72, 0.45, 0.22])

# Adicionar ruido realista (+/- 5%)
rs_ro_medido = rs_ro_cal * (1 + 0.05 * np.random.randn(len(rs_ro_cal)))

# --- Modelo 1: Lei de Potencia ---
# ppm = a * (Rs/Ro)^b
def modelo_potencia(x, a, b):
    return a * np.power(x, b)

popt_pot, pcov_pot = curve_fit(modelo_potencia, rs_ro_medido, ppm_cal,
                                p0=[100, -2.5])
a_pot, b_pot = popt_pot

# --- Modelo 2: Exponencial ---
# ppm = a * exp(b * Rs/Ro)
def modelo_exponencial(x, a, b):
    return a * np.exp(b * x)

popt_exp, pcov_exp = curve_fit(modelo_exponencial, rs_ro_medido, ppm_cal,
                                p0=[500, -1.5], maxfev=5000)
a_exp, b_exp = popt_exp

# --- Calculo de R-quadrado ---
def r_quadrado(y_real, y_pred):
    ss_res = np.sum((y_real - y_pred) ** 2)
    ss_tot = np.sum((y_real - np.mean(y_real)) ** 2)
    return 1 - ss_res / ss_tot

pred_pot = modelo_potencia(rs_ro_medido, *popt_pot)
pred_exp = modelo_exponencial(rs_ro_medido, *popt_exp)

r2_pot = r_quadrado(ppm_cal, pred_pot)
r2_exp = r_quadrado(ppm_cal, pred_exp)

erro_max_pot = np.max(np.abs(ppm_cal - pred_pot))
erro_max_exp = np.max(np.abs(ppm_cal - pred_exp))
erro_med_pot = np.mean(np.abs(ppm_cal - pred_pot))
erro_med_exp = np.mean(np.abs(ppm_cal - pred_exp))

print('=== Resultados da Calibracao ===')
print(f'\nModelo Lei de Potencia: ppm = {a_pot:.2f} * (Rs/Ro)^({b_pot:.3f})')
print(f'  R-quadrado: {r2_pot:.6f}')
print(f'  Erro maximo: {erro_max_pot:.2f} ppm')
print(f'  Erro medio:  {erro_med_pot:.2f} ppm')

print(f'\nModelo Exponencial: ppm = {a_exp:.2f} * exp({b_exp:.3f} * Rs/Ro)')
print(f'  R-quadrado: {r2_exp:.6f}')
print(f'  Erro maximo: {erro_max_exp:.2f} ppm')
print(f'  Erro medio:  {erro_med_exp:.2f} ppm')

melhor = 'Lei de Potencia' if r2_pot > r2_exp else 'Exponencial'
print(f'\nMelhor modelo: {melhor}')

In [ ]:
# Visualizacao das curvas de calibracao
x_fit = np.linspace(0.15, 4.0, 200)
y_pot = modelo_potencia(x_fit, *popt_pot)
y_exp = modelo_exponencial(x_fit, *popt_exp)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Curvas de calibracao (escala linear)
ax = axes[0, 0]
ax.scatter(rs_ro_medido, ppm_cal, c='red', s=80, zorder=5,
           edgecolors='black', label='Dados de calibracao')
ax.plot(x_fit, y_pot, 'b-', linewidth=2,
        label=f'Potencia (R2={r2_pot:.4f})')
ax.plot(x_fit, y_exp, 'g--', linewidth=2,
        label=f'Exponencial (R2={r2_exp:.4f})')
ax.set_xlabel('Rs/Ro')
ax.set_ylabel('Concentracao NH3 (ppm)')
ax.set_title('Curvas de Calibracao')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Escala log-log
ax = axes[0, 1]
ax.scatter(rs_ro_medido, ppm_cal, c='red', s=80, zorder=5,
           edgecolors='black', label='Dados')
ax.plot(x_fit, y_pot, 'b-', linewidth=2, label='Potencia')
ax.plot(x_fit, y_exp, 'g--', linewidth=2, label='Exponencial')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Rs/Ro (log)')
ax.set_ylabel('ppm (log)')
ax.set_title('Curvas de Calibracao (log-log)')
ax.legend()
ax.grid(True, which='both', alpha=0.3)

# 3. Residuos
ax = axes[1, 0]
residuos_pot = ppm_cal - pred_pot
residuos_exp = ppm_cal - pred_exp
x_pos = np.arange(len(ppm_cal))
width = 0.35
ax.bar(x_pos - width/2, residuos_pot, width, label='Potencia', color='blue', alpha=0.7)
ax.bar(x_pos + width/2, residuos_exp, width, label='Exponencial', color='green', alpha=0.7)
ax.set_xticks(x_pos)
ax.set_xticklabels([f'{p:.0f}' for p in ppm_cal])
ax.set_xlabel('Concentracao Real (ppm)')
ax.set_ylabel('Residuo (ppm)')
ax.set_title('Residuos por Ponto de Calibracao')
ax.axhline(y=0, color='black', linewidth=0.5)
ax.legend()
ax.grid(True, alpha=0.3)

# 4. Erro percentual
ax = axes[1, 1]
erro_pct_pot = 100 * np.abs(residuos_pot) / ppm_cal
erro_pct_exp = 100 * np.abs(residuos_exp) / ppm_cal
ax.plot(ppm_cal, erro_pct_pot, 'bo-', linewidth=2, label='Potencia')
ax.plot(ppm_cal, erro_pct_exp, 'gs--', linewidth=2, label='Exponencial')
ax.axhline(y=10, color='r', linestyle=':', alpha=0.7, label='Limiar 10%')
ax.set_xlabel('Concentracao Real (ppm)')
ax.set_ylabel('Erro Relativo (%)')
ax.set_title('Erro Percentual por Concentracao')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('calibracao_sensor.png', dpi=100, bbox_inches='tight')
plt.show()
print('Analise de calibracao concluida!')

In [ ]:
# Funcao final de conversao (usando melhor modelo)
def converter_rs_para_ppm(Rs, Ro, a=None, b=None):
    """Converte Rs/Ro em ppm usando lei de potencia calibrada."""
    if a is None:
        a = a_pot
    if b is None:
        b = b_pot
    ratio = Rs / Ro
    ppm = a * (ratio ** b)
    return max(0, min(ppm, 1000))  # Limitar faixa

# Demonstracao
print('Tabela de conversao (usando calibracao obtida):')
print(f'Parametros: a = {a_pot:.2f}, b = {b_pot:.3f}')
print('-' * 40)
print(f'{"Rs/Ro":>10} | {"ppm (calc)":>10} | {"ppm (real)":>10}')
print('-' * 40)
for rs_ro, ppm_real in zip(rs_ro_cal, ppm_cal):
    ppm_calc = converter_rs_para_ppm(rs_ro, 1.0)
    print(f'{rs_ro:>10.2f} | {ppm_calc:>10.1f} | {ppm_real:>10.0f}')

## Exercicios

1. **Curva de Aquecimento:** Simule a resposta transiente do sensor MOS
   durante o pre-aquecimento (cold start). Plote Rs vs tempo durante
   as primeiras 2 horas e determine o tempo minimo de estabilizacao.

2. **Array de Sensores:** Peca ao Claude para projetar um "nariz eletronico"
   com 3 sensores MOS diferentes (SnO2, WO3, ZnO) e implementar PCA
   (Analise de Componentes Principais) para discriminar NH3 de H2S.

3. **Compensacao Avancada:** Implemente uma rede neural simples (MLP com
   scikit-learn) que recebe Rs, temperatura e umidade como entrada e
   retorna a concentracao de NH3 compensada.

4. **Deriva do Sensor:** Simule a degradacao do sensor ao longo de 1 ano
   (aumento gradual de Ro). Implemente um algoritmo de auto-calibracao
   que detecte e compense essa deriva.

5. **Protocolo de Calibracao:** Peca ao Claude para escrever um procedimento
   operacional padrao (POP) para calibracao do sensor em campo, incluindo
   gases padrao, equipamentos e criterios de aceitacao.

## Proximos Passos

- Validar sensor com gases padrao em laboratorio
- Projetar enclosure (caixa) com protecao IP65 para uso em campo
- Implementar comunicacao LoRa para aviarios sem WiFi
- Integrar com sistema de ventilacao automatica do aviario
- Certificacao junto a orgaos reguladores (INMETRO, se aplicavel)
- Estudo de durabilidade em ambiente real (teste de 6 meses)

---

*Notebook criado com auxilio do Claude (Anthropic) para o modulo 13 - Enfitec Projetos.*